# Planning & Reasoning Evaluation: ReAct vs Direct

This notebook compares two agent reasoning approaches:
- **Direct**: Answer immediately without explicit reasoning
- **ReAct**: Reasoning + Acting with Thought → Action → Observation loops

In [ ]:
!pip install groq -q

In [ ]:
import getpass
import os
import re
from groq import Groq

os.environ["GROQ_API_KEY"] = getpass.getpass("Enter your GROQ API key: ")
client = Groq()

---
## 1. Simulated Tools

Simple tools the agent can use during ReAct reasoning.

In [ ]:
import math

def calculator(expression: str) -> str:
    """Evaluate a math expression"""
    try:
        result = eval(expression, {"__builtins__": {}}, {"math": math})
        return f"Result: {result}"
    except:
        return "Error: Invalid expression"

def search(query: str) -> str:
    """Simulated search with predefined knowledge"""
    knowledge = {
        "speed of light": "The speed of light is 299,792,458 meters per second (approximately 3 x 10^8 m/s).",
        "earth radius": "Earth's radius is approximately 6,371 kilometers.",
        "earth circumference": "Earth's circumference is approximately 40,075 kilometers.",
        "moon distance": "The Moon is approximately 384,400 kilometers from Earth.",
        "light year": "A light year is approximately 9.461 x 10^12 kilometers.",
        "population china": "China's population is approximately 1.4 billion people.",
        "population india": "India's population is approximately 1.4 billion people.",
        "amazon river length": "The Amazon River is approximately 6,400 kilometers long.",
        "nile river length": "The Nile River is approximately 6,650 kilometers long.",
        "everest height": "Mount Everest is 8,849 meters tall.",
        "mariana trench depth": "The Mariana Trench is approximately 10,994 meters deep."
    }
    query_lower = query.lower()
    for key, value in knowledge.items():
        if key in query_lower:
            return value
    return "No results found."

def execute_tool(action: str) -> str:
    """Parse and execute a tool call"""
    if action.startswith("calculator:"):
        expr = action.replace("calculator:", "").strip()
        return calculator(expr)
    elif action.startswith("search:"):
        query = action.replace("search:", "").strip()
        return search(query)
    else:
        return "Unknown action. Use calculator: <expr> or search: <query>"

print("Tools available:")
print("  - calculator: <expression>")
print("  - search: <query>")

---
## 2. Evaluation Metrics

We evaluate reasoning approaches on three metrics.

### Metric 1: Answer Correctness
**What it measures:** Is the final answer factually correct?

### Metric 2: Reasoning Quality  
**What it measures:** Did the agent show clear, logical reasoning steps?

### Metric 3: Tool Use Appropriateness
**What it measures:** Did the agent use tools when needed (and not when unnecessary)?

In [ ]:
NUM_TASKS = 5  # Set to 8 for full evaluation

def evaluate_response(task, response, reference, needs_tools):
    """Evaluate response using Llama 70B as judge"""
    
    eval_prompt = f"""Evaluate this AI agent response. Be STRICT (0=Fail, 1=Pass).

Task: {task}
Agent Response: {response}
Reference Answer: {reference}
Task Requires Tools: {needs_tools}

Evaluate:

1. Answer Correctness (0 or 1):
   Is the final answer correct? Check numbers and facts against reference.
   Score 0 if the answer is wrong or significantly incomplete.

2. Reasoning Quality (0 or 1):
   Did the agent show clear step-by-step reasoning?
   Score 1 if reasoning is logical and traceable, 0 if answer appears without explanation.

3. Tool Use Appropriateness (0 or 1):
   If task needed tools: Score 1 if tools were used correctly, 0 if not used or misused.
   If task didn't need tools: Score 1 if no unnecessary tool calls, 0 if tools were used unnecessarily.

Respond ONLY in this format:
Answer Correctness: X
Reasoning Quality: X
Tool Use Appropriateness: X"""

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": eval_prompt}],
        temperature=0
    )
    
    text = response.choices[0].message.content
    scores = {}
    for metric in ["Answer Correctness", "Reasoning Quality", "Tool Use Appropriateness"]:
        match = re.search(rf"{metric}\s*:\s*([01])", text, re.IGNORECASE)
        scores[metric] = int(match.group(1)) if match else 0
    return scores

---
## 3. Agent Approaches

### Approach 1: Direct (No Planning)
The agent answers immediately without explicit reasoning steps.

In [ ]:
def direct_agent(task: str) -> str:
    """Direct approach - answer without explicit reasoning"""
    prompt = f"""Answer this question directly and concisely.

Question: {task}

Answer:"""
    
    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.3
    )
    return response.choices[0].message.content

### Approach 2: ReAct (Reasoning + Acting)
The agent alternates between Thought, Action, and Observation steps.

In [ ]:
def react_agent(task: str, max_steps: int = 5) -> str:
    """ReAct approach - Thought/Action/Observation loop"""
    
    system_prompt = """You are a reasoning agent that solves problems step-by-step.

You have access to these tools:
- calculator: <expression> - Evaluate math expressions
- search: <query> - Look up factual information

Use this format:
Thought: <your reasoning about what to do next>
Action: <tool_name>: <input>

After receiving an Observation, continue with another Thought/Action or give your final answer:
Thought: <reasoning>
Final Answer: <your answer>

Always think before acting. Use tools when you need factual data or calculations."""

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": f"Task: {task}"}
    ]
    
    full_trace = ""
    
    for step in range(max_steps):
        response = client.chat.completions.create(
            model="llama-3.1-8b-instant",
            messages=messages,
            temperature=0.3,
            stop=["Observation:"]
        )
        
        agent_output = response.choices[0].message.content.strip()
        full_trace += agent_output + "\n"
        
        # Check if agent gave final answer
        if "Final Answer:" in agent_output:
            break
        
        # Extract action if present
        action_match = re.search(r"Action:\s*(.+)", agent_output)
        if action_match:
            action = action_match.group(1).strip()
            observation = execute_tool(action)
            full_trace += f"Observation: {observation}\n"
            
            # Add to conversation
            messages.append({"role": "assistant", "content": agent_output})
            messages.append({"role": "user", "content": f"Observation: {observation}"})
        else:
            # No action found, might be thinking or done
            break
    
    return full_trace

In [ ]:
approaches = {
    "Direct": direct_agent,
    "ReAct": react_agent
}

print("Approaches:")
for name in approaches:
    print(f"  - {name}")

---
## 4. Evaluation Tasks

Tasks that benefit from step-by-step reasoning and tool use.

In [ ]:
eval_tasks = [
    {
        "task": "How many seconds does it take for light to travel from Earth to the Moon?",
        "reference": "About 1.28 seconds. Moon is ~384,400 km away, light travels at ~300,000 km/s, so 384400/300000 = 1.28 seconds.",
        "needs_tools": True
    },
    {
        "task": "Which is longer: the Amazon River or the Nile River? By how many kilometers?",
        "reference": "The Nile is longer by about 250 km. Nile is ~6,650 km, Amazon is ~6,400 km.",
        "needs_tools": True
    },
    {
        "task": "If you walked around Earth's equator at 5 km/h, how many days would it take?",
        "reference": "About 334 days. Earth's circumference is ~40,075 km. At 5 km/h, that's 40075/5 = 8015 hours = ~334 days.",
        "needs_tools": True
    },
    {
        "task": "What is the total height difference between Mount Everest's peak and the bottom of the Mariana Trench?",
        "reference": "About 19,843 meters. Everest is 8,849m above sea level, Mariana Trench is 10,994m below. Total: 8849 + 10994 = 19,843m.",
        "needs_tools": True
    },
    {
        "task": "What is 15% of 847?",
        "reference": "127.05. Calculation: 847 * 0.15 = 127.05",
        "needs_tools": True
    },
    {
        "task": "A train travels 450 km in 3.5 hours. What is its average speed in km/h?",
        "reference": "About 128.57 km/h. Speed = distance/time = 450/3.5 = 128.57 km/h.",
        "needs_tools": True
    },
    {
        "task": "If China and India together have about 2.8 billion people, what percentage of the world's 8 billion population is that?",
        "reference": "35%. Calculation: (2.8/8) * 100 = 35%",
        "needs_tools": True
    },
    {
        "task": "How many times could you stack Mount Everest to reach the Moon?",
        "reference": "About 43,447 times. Moon is ~384,400 km = 384,400,000 m. Everest is 8,849 m. So 384400000/8849 = ~43,447.",
        "needs_tools": True
    }
]

print(f"Total tasks: {len(eval_tasks)}")
print(f"Testing with: {NUM_TASKS} tasks")

---
## 5. Compare Sample Outputs

Let's see how each approach handles the same task.

In [ ]:
sample_task = eval_tasks[0]["task"]
print(f"Task: {sample_task}")
print("=" * 70)

for approach_name, agent_fn in approaches.items():
    print(f"\n{approach_name} Approach:")
    print("-" * 50)
    output = agent_fn(sample_task)
    print(output)

print("\n" + "=" * 70)
print(f"Reference: {eval_tasks[0]['reference']}")

---
## 6. Run Evaluation

In [ ]:
results = {name: {"Answer Correctness": [], "Reasoning Quality": [], "Tool Use Appropriateness": []}
           for name in approaches}

test_tasks = eval_tasks[:NUM_TASKS]

for approach_name, agent_fn in approaches.items():
    print(f"\nEvaluating: {approach_name}")
    print("=" * 50)
    
    for i, item in enumerate(test_tasks):
        # Generate response
        response = agent_fn(item["task"])
        
        # Evaluate
        scores = evaluate_response(
            item["task"], 
            response, 
            item["reference"],
            item["needs_tools"]
        )
        
        for metric, score in scores.items():
            results[approach_name][metric].append(score)
        
        print(f"\nTask {i+1}: {item['task'][:50]}...")
        print(f"Scores: {scores}")

print(f"\n(Evaluated {NUM_TASKS} tasks per approach)")

---
## 7. Results Summary

In [ ]:
print("\n" + "="*70)
print("PLANNING & REASONING EVALUATION RESULTS")
print("="*70)
print(f"{'Approach':<15} {'Correct':>12} {'Reasoning':>12} {'Tool Use':>12} {'Overall':>12}")
print("-"*70)

best_approach = None
best_score = 0

for approach_name in approaches:
    n = len(results[approach_name]["Answer Correctness"])
    correct = sum(results[approach_name]["Answer Correctness"]) / n
    reasoning = sum(results[approach_name]["Reasoning Quality"]) / n
    tool_use = sum(results[approach_name]["Tool Use Appropriateness"]) / n
    overall = (correct + reasoning + tool_use) / 3
    
    print(f"{approach_name:<15} {correct:>12.0%} {reasoning:>12.0%} {tool_use:>12.0%} {overall:>12.0%}")
    
    if overall > best_score:
        best_score = overall
        best_approach = approach_name

print("-"*70)
print(f"Best: {best_approach} ({best_score:.0%})")

---
## 8. Summary

### ReAct (Reasoning + Acting)

```
Thought: I need to find the distance to the Moon
Action: search: moon distance
Observation: The Moon is approximately 384,400 km from Earth
Thought: Now I need to calculate time = distance / speed of light
Action: calculator: 384400 / 300000
Observation: Result: 1.28
Thought: I have the answer
Final Answer: Light takes about 1.28 seconds to reach the Moon
```

### Approach Comparison

| Approach | Pros | Cons |
|----------|------|------|
| Direct | Fast, simple | No reasoning trace, may hallucinate facts |
| ReAct | Explicit reasoning, uses tools, verifiable | More tokens, slower |

### Key Insights

1. **Answer Correctness**: ReAct produces more accurate answers by retrieving facts rather than relying on parametric memory

2. **Reasoning Quality**: ReAct provides explicit thought traces that can be audited and debugged

3. **Tool Use**: ReAct appropriately uses tools for calculations and lookups; Direct approach often guesses

### Takeaway

For tasks requiring factual accuracy and multi-step reasoning, ReAct's explicit Thought→Action→Observation loop outperforms direct answering. The reasoning trace also makes the agent's behavior interpretable and debuggable.